In [4]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [5]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [6]:
openai = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL")
)

In [7]:
# Some lists!

todos = []
completed = []

In [8]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [10]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [11]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [12]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
Plan a 3-step morning routine and execute it
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Create a simple 3-step morning routine with clear actions and time estimates
Todo #2: "Execute" the routine by simulating completion of each step with brief notes
Todo #3: Present the completed routine in Rich console markup (no code blocks)

Drafted a 3-step routine: hydrate + light stretch (5–7 min), hygiene + get dressed (10–15 min), plan day + quick 
breakfast (15–20 min).

Todo #1: Create a simple 3-step morning routine with clear actions and time estimates
Todo #2: "Execute" the routine by simulating completion of each step with brief notes
Todo #3: Present the completed routine in Rich console markup (no code blocks)

Simulated completion: drank water, did mobility; brushed/washed/dressed; reviewed top 3 priorities and ate a simple
breakfast.

Todo #1: Create a simple 3-step morning routine with clear actions and time estimates
Todo #2: "Execute" the routine by simulating completion of each step with brief notes
Todo #3: Present the completed routine in Rich console markup (no code blocks)

Prepared final formatted output in Rich markup with steps, timings, and execution log.

Todo #1: Create a simple 3-step morning routine with clear actions and time estimates
Todo #2: "Execute" the routine by simulating completion of each step with brief notes
Todo #3: Present the completed routine in Rich console markup (no code blocks)

3-Step Morning Routine (with execution)

Step 1 — Wake + Hydrate + Mobility (5–7 min)
• Drink a full glass of water  
• 2 minutes of easy stretching (neck/shoulders/hips)  
• 10 slow breaths to fully “wake up”  
Executed: Drank water, did light mobility, finished with calm breathing.

Step 2 — Hygiene + Get Ready (10–15 min)
• Bathroom + wash face  
• Brush teeth  
• Get dressed (set out simple outfit)  
Executed: Washed up, brushed teeth, got dressed.

Step 3 — Plan + Fuel (15–20 min)
• Review calendar + choose Top 3 priorities for the day  
• Quick breakfast (e.g., yogurt + fruit / eggs + toast / oatmeal)  
• Pack essentials (keys/wallet/phone, water)  
Executed: Picked top 3 tasks, ate a simple breakfast, and gathered essentials.

Total time: ~30–40 minutes